<a href="https://colab.research.google.com/github/i2mmmmm/train_project/blob/main/20230818_ar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LayerNormalization, MultiHeadAttention, Flatten
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler


In [4]:
from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
s30_trail = pd.read_csv('/content/drive/My Drive/철도/s30_trail.csv')
s40_trail = pd.read_csv('/content/drive/My Drive/철도/s40_trail.csv')
s50_trail = pd.read_csv('/content/drive/My Drive/철도/s50_trail.csv')
s70_trail = pd.read_csv('/content/drive/My Drive/철도/s70_trail.csv')
s100_trail = pd.read_csv('/content/drive/My Drive/철도/s100_trail.csv')
c30_trail = pd.read_csv('/content/drive/My Drive/철도/c30_trail.csv')
c40_trail = pd.read_csv('/content/drive/My Drive/철도/c40_trail.csv')
c50_trail = pd.read_csv('/content/drive/My Drive/철도/c50_trail.csv')
c70_trail = pd.read_csv('/content/drive/My Drive/철도/c70_trail.csv')
c100_trail = pd.read_csv('/content/drive/My Drive/철도/c100_trail.csv')


In [29]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)

data_s_X, data_s_y = prepare_data(c100_trail)

N_TIMESTEPS = 10
N_FEATURES = 34

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(4))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

# 예측 결과 출력
print("Predictions for data_s")
print(predictions_s[:5])



Epoch 1/50
157/157 [==============================] - 35s 142ms/step - loss: 8.5879e-04 - mae: 0.0203 - val_loss: 5.4739e-04 - val_mae: 0.0171
Epoch 2/50
157/157 [==============================] - 16s 102ms/step - loss: 2.3459e-04 - mae: 0.0116 - val_loss: 5.6547e-04 - val_mae: 0.0172
Epoch 3/50
157/157 [==============================] - 16s 102ms/step - loss: 1.8054e-04 - mae: 0.0101 - val_loss: 5.9933e-04 - val_mae: 0.0180
Epoch 4/50
157/157 [==============================] - 16s 101ms/step - loss: 1.5972e-04 - mae: 0.0095 - val_loss: 5.4840e-04 - val_mae: 0.0172
Epoch 5/50
157/157 [==============================] - 16s 101ms/step - loss: 1.4475e-04 - mae: 0.0090 - val_loss: 6.3225e-04 - val_mae: 0.0182
Epoch 6/50
157/157 [==============================] - 16s 101ms/step - loss: 1.3170e-04 - mae: 0.0085 - val_loss: 6.9684e-04 - val_mae: 0.0192
Epoch 7/50
157/157 [==============================] - 16s 104ms/step - loss: 1.2362e-04 - mae: 0.0083 - val_loss: 6.4833e-04 - val_mae: 0.0183

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 예측과 실제값 간의 MAE, MSE, RMSE, R-squared 계산
mae = mean_absolute_error(y_test_s, predictions_s)
mse = mean_squared_error(y_test_s, predictions_s)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_s, predictions_s)

# 결과 출력
print("Evaluation metrics for data_s:")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R-squared:", r2)


Evaluation metrics for data_s:
MAE: 0.004689485690516887
MSE: 5.018317484975397e-05
RMSE: 0.007084008388599915
R-squared: 0.9032700239745032


In [11]:
predictions_30s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_30s.to_csv('/content/drive/My Drive/철도/30s_20230818_ar.csv', index=False)

In [13]:
predictions_40s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_40s.to_csv('/content/drive/My Drive/철도/40s_20230818_ar.csv', index=False)

In [15]:
predictions_50s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_50s.to_csv('/content/drive/My Drive/철도/50s_20230818_ar.csv', index=False)

In [17]:
predictions_70s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_70s.to_csv('/content/drive/My Drive/철도/70s_20230818_ar.csv', index=False)

In [19]:
predictions_100s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_100s.to_csv('/content/drive/My Drive/철도/100s_20230818_ar.csv', index=False)

In [22]:
predictions_30c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_30c.to_csv('/content/drive/My Drive/철도/30c_20230818_ar.csv', index=False)

In [24]:
predictions_40c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_40c.to_csv('/content/drive/My Drive/철도/40c_20230818_ar.csv', index=False)

In [26]:
predictions_50c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_50c.to_csv('/content/drive/My Drive/철도/50c_20230818_ar.csv', index=False)

In [28]:
predictions_70c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_70c.to_csv('/content/drive/My Drive/철도/70c_20230818_ar.csv', index=False)

In [30]:
predictions_100c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_100c.to_csv('/content/drive/My Drive/철도/100c_20230818_ar.csv', index=False)

In [31]:
answer_sample = pd.read_csv('/content/drive/My Drive/철도/answer_sample.csv')
c30 = pd.read_csv('/content/drive/My Drive/철도/30c_20230818_ar.csv')
c40 = pd.read_csv('/content/drive/My Drive/철도/40c_20230818_ar.csv')
c50 = pd.read_csv('/content/drive/My Drive/철도/50c_20230818_ar.csv')
c70 = pd.read_csv('/content/drive/My Drive/철도/70c_20230818_ar.csv')
c100 = pd.read_csv('/content/drive/My Drive/철도/100c_20230818_ar.csv')
s30 = pd.read_csv('/content/drive/My Drive/철도/30s_20230818_ar.csv')
s40 = pd.read_csv('/content/drive/My Drive/철도/40s_20230818_ar.csv')
s50 = pd.read_csv('/content/drive/My Drive/철도/50s_20230818_ar.csv')
s70 = pd.read_csv('/content/drive/My Drive/철도/70s_20230818_ar.csv')
s100 = pd.read_csv('/content/drive/My Drive/철도/100s_20230818_ar.csv')

In [32]:
c30.columns = ['YL_M1_B1_W1_c30', 'YR_M1_B1_W1_c30', 'YL_M1_B1_W2_c30', 'YR_M1_B1_W2_c30']
c40.columns = ['YL_M1_B1_W1_c40', 'YR_M1_B1_W1_c40', 'YL_M1_B1_W2_c40', 'YR_M1_B1_W2_c40']
c50.columns = ['YL_M1_B1_W1_c50', 'YR_M1_B1_W1_c50', 'YL_M1_B1_W2_c50', 'YR_M1_B1_W2_c50']
c70.columns = ['YL_M1_B1_W1_c70', 'YR_M1_B1_W1_c70', 'YL_M1_B1_W2_c70', 'YR_M1_B1_W2_c70']
c100.columns = ['YL_M1_B1_W1_c100', 'YR_M1_B1_W1_c100', 'YL_M1_B1_W2_c100', 'YR_M1_B1_W2_c100']

s30.columns = ['YL_M1_B1_W1_s30', 'YR_M1_B1_W1_s30', 'YL_M1_B1_W2_s30', 'YR_M1_B1_W2_s30']
s40.columns = ['YL_M1_B1_W1_s40', 'YR_M1_B1_W1_s40', 'YL_M1_B1_W2_s40', 'YR_M1_B1_W2_s40']
s50.columns = ['YL_M1_B1_W1_s50', 'YR_M1_B1_W1_s50', 'YL_M1_B1_W2_s50', 'YR_M1_B1_W2_s50']
s70.columns = ['YL_M1_B1_W1_s70', 'YR_M1_B1_W1_s70', 'YL_M1_B1_W2_s70', 'YR_M1_B1_W2_s70']
s100.columns = ['YL_M1_B1_W1_s100', 'YR_M1_B1_W1_s100', 'YL_M1_B1_W2_s100', 'YR_M1_B1_W2_s100']

In [33]:
answer = pd.concat([answer_sample.Distance, s30,s40,s50,s70,s100,c30,c40,c50,c70,c100], axis=1)

In [34]:
answer

,Distance,YL_M1_B1_W1_s30,YR_M1_B1_W1_s30,YL_M1_B1_W2_s30,YR_M1_B1_W2_s30,YL_M1_B1_W1_s40,YR_M1_B1_W1_s40,YL_M1_B1_W2_s40,YR_M1_B1_W2_s40,YL_M1_B1_W1_s50,...,YL_M1_B1_W2_c50,YR_M1_B1_W2_c50,YL_M1_B1_W1_c70,YR_M1_B1_W1_c70,YL_M1_B1_W2_c70,YR_M1_B1_W2_c70,YL_M1_B1_W1_c100,YR_M1_B1_W1_c100,YL_M1_B1_W2_c100,YR_M1_B1_W2_c100
0,2500.25,0.022727,-0.007023,0.051396,-0.041762,0.000310,0.006870,0.002972,0.002070,0.006523,...,-0.004512,0.011268,0.000782,0.008793,-0.001240,0.007939,0.003591,0.005443,-0.003415,0.011239
1,2500.50,0.021518,-0.002455,0.057657,-0.047653,-0.004463,0.012639,0.007021,-0.002991,-0.000533,...,0.002812,0.003643,-0.003370,0.012878,0.006945,-0.000688,-0.000737,0.009284,0.003571,0.003431
2,2500.75,0.014678,0.008294,0.061176,-0.051550,-0.013558,0.022481,0.010421,-0.007791,-0.013033,...,0.013031,-0.007091,-0.012010,0.021638,0.016372,-0.010743,-0.009563,0.017688,0.012526,-0.006470
3,2501.00,0.008154,0.017232,0.059993,-0.051226,-0.022525,0.031701,0.011702,-0.010190,-0.024127,...,0.023015,-0.017845,-0.020773,0.030647,0.023441,-0.018423,-0.017691,0.025551,0.021405,-0.016433
4,2501.25,0.005902,0.019330,0.057320,-0.049022,-0.025714,0.034916,0.011050,-0.009626,-0.027441,...,0.031793,-0.027147,-0.024005,0.034067,0.027369,-0.022578,-0.020815,0.028721,0.028200,-0.023836
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1994,2998.75,0.011671,-0.000941,0.047094,-0.037910,0.013337,-0.003593,0.040749,-0.030107,0.012479,...,0.047127,-0.039867,0.012437,-0.004077,0.033943,-0.026843,0.013455,-0.004137,0.041506,-0.031546
1995,2999.00,0.011528,-0.001421,0.040249,-0.030580,0.010641,-0.002050,0.036552,-0.026102,0.008692,...,0.042169,-0.034676,0.011651,-0.003036,0.032324,-0.024109,0.009844,-0.001344,0.038351,-0.027754
1996,2999.25,0.007805,0.002062,0.025199,-0.016347,0.004760,0.002338,0.024174,-0.015295,0.001605,...,0.027416,-0.020774,0.006089,0.002873,0.021705,-0.013847,0.004455,0.003260,0.027638,-0.018120
1997,2999.50,0.007623,0.001276,0.009979,-0.002355,0.004618,0.001550,0.008413,-0.000919,0.003754,...,0.008342,-0.003247,0.003375,0.005797,0.006655,-0.000009,0.003931,0.003246,0.013773,-0.005947


In [35]:
answer.to_csv('/content/drive/My Drive/철도/answer20230818_ar.csv', index=False)